# Explore MSD Datasets

This notebook uses the standard PyHealth flow: dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. `show_msd_samples()` can either show image and mask separately or overlay the mask on the image. Data is read from `PYHEALTH_IMAGING_DATA` (default `~/pyhealth-imaging-data`); `download=True` fetches missing MSD tasks automatically.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import (
    MSDHippocampusDataset,
    MSDProstateDataset,
    MSDSpleenDataset,
    get_dataloader,
)

DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
# Missing MSD tasks are downloaded into DATA_ROOT when download=True.
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = Path(CACHE_DIR) / "metadata"

DATASETS = {
    "hippocampus": (MSDHippocampusDataset, DATA_ROOT / "Task04_Hippocampus"),
    "prostate": (MSDProstateDataset, DATA_ROOT / "Task05_Prostate"),
    "spleen": (MSDSpleenDataset, DATA_ROOT / "Task09_Spleen"),
}


def load_msd_samples(name):
    DatasetClass, root = DATASETS[name]
    dataset = DatasetClass(
        root=root,
        cache_dir=CACHE_DIR,
        metadata_dir=str(Path(METADATA_DIR) / name),
        download=True,
    )
    dataset.stats()
    samples = dataset.set_task(dataset.default_task, num_workers=1)
    sample = samples[0]
    print("samples:", len(samples))
    print("input schema:", samples.input_schema)
    print("output schema:", samples.output_schema)
    print("image:", sample["image"].shape, sample["image"].dtype)
    print("mask:", sample["mask"].shape, sample["mask"].dtype)
    return dataset, samples


def show_msd_samples(samples, n=3, overlay_mask=False, channel=0):
    n = min(n, len(samples))
    rows = 1 if overlay_mask else 2
    fig, axes = plt.subplots(rows, n, figsize=(4 * n, 4 * rows), squeeze=False)
    for col in range(n):
        sample = samples[col]
        slice_idx = sample["image"].shape[1] // 2
        image = sample["image"][channel, slice_idx].cpu()
        mask = sample["mask"][0, slice_idx].cpu()
        channel_label = f" ch{channel}" if sample["image"].shape[0] > 1 else ""
        if overlay_mask:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
            axes[0, col].set_title(f"{sample['patient_id']}{channel_label}")
        else:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].set_title(f"{sample['patient_id']}{channel_label}")
            axes[1, col].imshow(mask, cmap="gray")
            axes[1, col].set_title("mask")
        for row in range(rows):
            axes[row, col].axis("off")
    plt.show()


## Hippocampus

Mask is overlaid on the image.

In [ ]:
hippo_dataset, hippo_samples = load_msd_samples("hippocampus")
show_msd_samples(hippo_samples, n=3, overlay_mask=True)


In [ ]:
loader = get_dataloader(hippo_samples, batch_size=1)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch mask:", batch["mask"].shape)


## Prostate

Prostate has two channels (T2, ADC); the plot uses channel 0.

In [ ]:
prostate_dataset, prostate_samples = load_msd_samples("prostate")
show_msd_samples(prostate_samples, n=3, overlay_mask=True, channel=0)


## Spleen

Mask is overlaid on the CT image.

In [ ]:
spleen_dataset, spleen_samples = load_msd_samples("spleen")
show_msd_samples(spleen_samples, n=3, overlay_mask=True)


Set `overlay_mask=False` in any `show_msd_samples()` call to display the image and mask as separate rows instead.